[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/04_cell_specific_multisweep_fitting.ipynb)

In [ ]:
from pathlib import Path
import os, sys, subprocess


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


if _running_in_colab():
    repo_url = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
    repo_branch = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
    project_root = Path("/content") / "astromodel_proving"
    if not project_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(project_root)], check=True)
    os.chdir(project_root)
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next((c for c in candidates if (c / "src").is_dir() and (c / "data").is_dir()), current)
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")

# Step 04 — Cell-specific six-sweep fitting and accepted ensemble construction

This notebook fits one shared parameter vector per ATF file across its six ordered pump-current sweeps. It is the first step that builds **cell-specific accepted ensembles** rather than relying on historical single-current Optuna fits. The notebook uses region-aware, leave-one-cell-out empirical thresholds and reliability weights when deciding whether a candidate is accepted.

It constructs accepted ensembles but **does not yet claim mechanism diversity or biological degeneracy**. Those questions belong to later mechanism decomposition and predictive validation steps.

Model contract: this notebook uses `src.astro_model.model`, which implements the same 4-state ODE system discussed with reviewers (`Va, DK_a_t, K_a_s, Kg`) and the same equations for `K_o`, `E_k_a`, `I_Kir`, `I_kgap`, `I_l_a`, and the sigmoid/tanh/hill switching gate. Steps 00, 01, and 03 also use that same central model when they need simulation-based evaluation; Step 02 is purely experimental/feature-threshold reconstruction and does not use the ODE model.

In [ ]:
import os
import json
from pathlib import Path

import matplotlib.pyplot as plt
plt.switch_backend("Agg")
import pandas as pd
from IPython.display import display

from src.step04_cell_specific_multisweep import Step04Config, run_step04_cell_specific_multisweep, benchmark_step04_configs, OUTPUT_SUBDIR
from src.astro_model import VALID_CURRENTS, simulate_voltage_trace

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT = Path(os.environ["ASTROMODEL_PROJECT_ROOT"])
max_cells_env = os.environ.get("ASTROMODEL_STEP04_MAX_CELLS", "6")
config = Step04Config(
    random_seed=int(os.environ.get("ASTROMODEL_STEP04_RANDOM_SEED", "7")),
    n_candidates=int(os.environ.get("ASTROMODEL_STEP04_N_CANDIDATES", "4")),
    max_cells=None if max_cells_env.lower() in {"none", "all"} else int(max_cells_env),
    cell_selection_mode=os.environ.get("ASTROMODEL_STEP04_CELL_SELECTION_MODE", "group_balanced"),
    max_accepted_per_cell=int(os.environ.get("ASTROMODEL_STEP04_MAX_ACCEPTED_PER_CELL", "3")),
    write_outputs=True,
)
print(config)
results = run_step04_cell_specific_multisweep(PROJECT_ROOT, config)
out_dir = PROJECT_ROOT / "outputs" / OUTPUT_SUBDIR
print("output_dir=", out_dir)
print(json.dumps(results["analysis_summary"], indent=2))
fit_status = results["fit_status_by_cell"]
accepted_summary = results["accepted_ensemble_summary"]
accepted_candidates = results["accepted_candidates"]
display(fit_status.head(20))
display(accepted_summary)

In [ ]:
summary = accepted_summary.copy()
summary["accepted_fraction"] = summary["n_accepted_cells"] / summary["n_cells"]
fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{r.condition}-{r.region}" for r in summary.itertuples()]
ax.bar(labels, summary["accepted_fraction"])
ax.set_ylabel("Accepted fraction")
ax.set_ylim(0, 1)
ax.set_title("Accepted cells by condition and region")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, summary["median_best_trace_nrmse"])
ax.set_ylabel("Median best trace NRMSE")
ax.set_title("Best trace agreement by condition and region")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
cell_dict = results["cell_dict"]
accepted_only = fit_status[fit_status["status"] == "accepted"].copy()
representatives = (
    accepted_only.sort_values(["condition", "region", "best_objective"])
    .groupby("condition", as_index=False)
    .head(1)
)
print(representatives[["file_id", "region", "condition", "best_objective", "best_mean_weighted_pass_fraction"]].to_string(index=False))

plot_points = int(os.environ.get("ASTROMODEL_STEP04_PLOT_POINTS", "200"))
max_rep_plots = int(os.environ.get("ASTROMODEL_STEP04_MAX_REP_PLOTS", "1"))
for rep in representatives.head(max_rep_plots).itertuples():
    cell = cell_dict[rep.file_id]
    candidate = accepted_candidates[(accepted_candidates["file_id"] == rep.file_id) & (accepted_candidates["candidate_id"] == rep.best_candidate_id)].iloc[0].to_dict()
    fig, ax = plt.subplots(figsize=(8, 4))
    for sweep_idx, current_na in enumerate(VALID_CURRENTS, start=1):
        sweep = cell["sweeps"].get(sweep_idx)
        if sweep is None:
            continue
        obs_t_full = sweep["time_s"]
        obs_v_full = sweep["vm_mV"]
        plot_t = __import__("numpy").linspace(float(obs_t_full[0]), float(obs_t_full[-1]), plot_points)
        obs_v = __import__("numpy").interp(plot_t, obs_t_full, obs_v_full)
        onset_s = float(sweep["features"]["stim_onset_s"])
        offset_s = float(sweep["features"]["stim_offset_s"])
        sim_v = simulate_voltage_trace(cell["meta"]["condition"], current_na, candidate, plot_t * 1000.0, onset_s * 1000.0, offset_s * 1000.0)
        obs_baseline = __import__("numpy").median(obs_v[(plot_t >= max(plot_t[0], onset_s - 5.0)) & (plot_t < max(plot_t[0], onset_s - 1.0))])
        sim_baseline = __import__("numpy").median(sim_v[(plot_t >= max(plot_t[0], onset_s - 5.0)) & (plot_t < max(plot_t[0], onset_s - 1.0))])
        ax.plot(plot_t, obs_v - obs_baseline, alpha=0.45, label=f"obs {current_na} nA" if sweep_idx == 1 else None)
        ax.plot(plot_t, sim_v - sim_baseline, linewidth=1.1, label=f"fit {current_na} nA" if sweep_idx == 1 else None)
    ax.set_title(f"{rep.file_id} ({rep.condition}, {rep.region})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Baseline-centered Vm (mV)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
run_benchmark = os.environ.get("ASTROMODEL_STEP04_RUN_BENCHMARK", "0").lower() in {"1", "true", "yes"}
if run_benchmark:
    benchmark = benchmark_step04_configs(PROJECT_ROOT, config)
    display(benchmark)
    benchmark.to_csv(out_dir / "performance_benchmark.csv", index=False)
else:
    benchmark_path = out_dir / "performance_benchmark.csv"
    if benchmark_path.exists():
        benchmark = pd.read_csv(benchmark_path)
        display(benchmark)
    else:
        print("Benchmark skipped in fast execution mode.")


## Interpretation

This notebook shows whether one shared parameter vector can reproduce the six ordered current sweeps of a cell while satisfying region-aware empirical feature contracts. The output is a **cell-specific accepted ensemble**. This is stronger than historical single-current fitting, but it is still not the stage where mechanism diversity or biological degeneracy is claimed. Those questions depend on later mechanism decomposition and predictive validation.